# Section-level dense retrieval with FlagEmbedding BGE-M3

Test BGE-M3 through the official `BGEM3FlagModel` API. Sections may contain multiple legal articles. Retrieval is dense-only; returned sections are expanded back to all contained article IDs.

In [ ]:
import json
import math
from collections import OrderedDict
from pathlib import Path

import numpy as np
from FlagEmbedding import BGEM3FlagModel
from transformers import AutoTokenizer

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'data-pipeline').is_dir())
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
INPUT_PATH = PROCESSED_DIR / 'articles_normalized.jsonl'
EVALUATION_PATH = PROJECT_ROOT / 'data-pipeline' / 'evaluation' / 'data' / 'relevance_mapping.json'
OUTPUT_DIR = PROCESSED_DIR / 'test' / 'section_dense_bge_m3'
MODEL_NAME = 'BAAI/bge-m3'
MAX_SECTION_TOKENS = 7500
CANDIDATE_SECTIONS = 100
TOP_K_ARTICLES = 5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = BGEM3FlagModel(MODEL_NAME, use_fp16=False)

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def token_count(text):
    return len(tokenizer.encode(text, add_special_tokens=True))

articles = load_jsonl(INPUT_PATH)

def group_key(article):
    document_id = str(article.get('document_id'))
    chapter = article.get('chapter_number') or article.get('chapter_title') or 'no-chapter'
    section = article.get('section_number') or article.get('section_title')
    if section:
        return (document_id, str(chapter), str(section))
    if chapter != 'no-chapter':
        return (document_id, str(chapter), 'chapter-without-section')
    return (document_id, 'article-only', str(article['id']))

def article_block(article):
    label = f"មាត្រា {article.get('article_number')}"
    return f"{label}\n{article.get('search_text', article.get('text', ''))}".strip()

groups = OrderedDict()
for article in articles:
    groups.setdefault(group_key(article), []).append(article)

section_records = []
for section_key, group in groups.items():
    current_text, current_articles, chunk_index = '', [], 1
    for article in group:
        block = article_block(article)
        candidate = f'{current_text}\n\n{block}'.strip() if current_text else block
        if current_text and token_count(candidate) > MAX_SECTION_TOKENS:
            section_records.append({'section_id': f"{'|'.join(section_key)}-{chunk_index}", 'section_key': list(section_key), 'article_ids': list(current_articles), 'text': current_text, 'chunk_index': chunk_index})
            chunk_index += 1; current_text, current_articles = block, [str(article['article_id'])]
        else:
            current_text = candidate; current_articles.append(str(article['article_id']))
    if current_text:
        section_records.append({'section_id': f"{'|'.join(section_key)}-{chunk_index}", 'section_key': list(section_key), 'article_ids': list(current_articles), 'text': current_text, 'chunk_index': chunk_index})

texts = [record['text'] for record in section_records]
encoded = model.encode(texts, batch_size=2, max_length=8192, return_dense=True, return_sparse=False, return_colbert_vecs=False)
vectors = np.asarray(encoded['dense_vecs'], dtype='float32')
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.save(OUTPUT_DIR / 'section_vectors.npy', vectors)
(OUTPUT_DIR / 'sections.jsonl').write_text(''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in section_records), encoding='utf-8')
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps({'model': MODEL_NAME, 'api': 'FlagEmbedding.BGEM3FlagModel', 'dense_only': True, 'max_section_tokens': MAX_SECTION_TOKENS, 'section_count': len(section_records), 'article_count': len(articles), 'embedding_dimension': int(vectors.shape[1]), 'normalized_embeddings': True}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

def normalize_query(query):
    return ' '.join(str(query).replace('\u200b', '').replace('\u200c', '').replace('\u200d', '').split())

def search(query):
    encoded_query = model.encode([normalize_query(query)], max_length=8192, return_dense=True, return_sparse=False, return_colbert_vecs=False)
    query_vector = np.asarray(encoded_query['dense_vecs'][0], dtype='float32')
    query_vector /= np.linalg.norm(query_vector)
    scores = vectors @ query_vector
    article_scores = {}
    section_hits = []
    for index in np.argsort(-scores)[:CANDIDATE_SECTIONS]:
        section = section_records[index]
        score = float(scores[index])
        section_hits.append({'section_id': section['section_id'], 'score': score, 'article_ids': section['article_ids']})
        for article_id in section['article_ids']:
            article_scores[article_id] = max(article_scores.get(article_id, -1.0), score)
    ranked_articles = sorted(article_scores, key=article_scores.get, reverse=True)
    return ranked_articles[:TOP_K_ARTICLES], section_hits[:5]

mapping = json.loads(EVALUATION_PATH.read_text(encoding='utf-8'))
qrels = {item['query_id']: {article_id: 2 for article_id in item['relevant_articles']} for item in mapping}
queries = {item['query_id']: item['query'] for item in mapping}
available = {str(article['article_id']) for article in articles}
eligible = {qid: labels for qid, labels in qrels.items() if set(labels) & available}

def ndcg(ranked, labels, k=3):
    gain = lambda relevance: (2 ** relevance) - 1
    dcg = sum(gain(labels.get(article_id, 0)) / math.log2(index + 2) for index, article_id in enumerate(ranked[:k]))
    ideal = sorted((gain(value) for value in labels.values()), reverse=True)[:k]
    idcg = sum(value / math.log2(index + 2) for index, value in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

rows = []
for query_id, labels in eligible.items():
    ranked, sections = search(queries[query_id])
    ranks = [index + 1 for index, article_id in enumerate(ranked) if article_id in labels]
    rank = ranks[0] if ranks else None
    rows.append({'query_id': query_id, 'expected': labels, 'top_articles': ranked, 'top_sections': sections, 'rank': rank, 'hit@3': int(rank is not None and rank <= 3), 'hit@5': int(rank is not None and rank <= 5), 'mrr@3': 1 / rank if rank and rank <= 3 else 0.0, 'ndcg@3': ndcg(ranked, labels)})

metrics = {key: round(sum(row[key] for row in rows) / len(rows), 4) for key in ('hit@3', 'hit@5', 'mrr@3', 'ndcg@3')}
report = {'model': MODEL_NAME, 'api': 'FlagEmbedding.BGEM3FlagModel', 'retrieval': 'dense-only', 'max_section_tokens': MAX_SECTION_TOKENS, 'section_count': len(section_records), 'query_count': len(qrels), 'eligible_query_count': len(eligible), 'excluded_query_ids': sorted(set(qrels) - set(eligible)), 'metrics': metrics, 'results': rows}
(OUTPUT_DIR / 'evaluation.json').write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps({key: report[key] for key in ('model', 'api', 'retrieval', 'max_section_tokens', 'section_count', 'query_count', 'eligible_query_count', 'metrics')}, ensure_ascii=False, indent=2))
print('Saved:', OUTPUT_DIR / 'evaluation.json')